# Nonword orthographic ill-formedness

Two kinds of score per nonword, each averaged over the list:

- **letter n-gram** (`orthographic_ngram`): mean over the letters and the word end of −ln P(letter | previous
  1 or 2 letters), with counts from CMUdict spellings. Nats per letter; higher = less English-like.
- **neighbourhood** (`orthographic_neighbourhood`): every CMUdict word w weighted by
  P(w | x) ∝ f(w) e^(−β d), where d = letter edit distance ÷ the longer length and f = SUBTLEX-US count + 1.
  *Surprisal* (higher = far from every frequent word) and *entropy* (low = one neighbour dominates, high = many
  equally close), β = 10 and 20, plus OLD20.

Real words from the example sentences serve as a reference. Needs CMUdict
(`python -c "import nltk; nltk.download('cmudict')"`) and, for the neighbourhood, `linguistic fetch subtlex_us`.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from linguistic.assemble import run_metrics
from linguistic.io import load_stimuli
from linguistic.normalize import Lexicon, build_units
from linguistic.registry import Context, select

NONWORDS = ROOT / "data/examples/nonwords/stimuli.csv"
REFERENCE = ROOT / "data/examples/sentences/stimuli.csv"     # real words, for comparison

lexicon = Lexicon()

def orthographic(path):
    stimuli = load_stimuli(path)
    units = build_units(stimuli, lexicon)
    return run_metrics(Context(stimuli, units.words, units.phones, lexicon), select(["orthographic"]))

nw, ref = orthographic(NONWORDS), orthographic(REFERENCE)
SCORES = ["orth_trigram_surp", "orth_old20", "orth_nbhd_surprisal_b20", "orth_nbhd_entropy_b20"]
nw.words[["stimulus_id", "word", *SCORES]].sort_values("orth_nbhd_surprisal_b20").round(2)

,stimulus_id,word,orth_trigram_surp,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20
21,nonword_list_2,honude,2.88,1.95,13.44,3.02
56,nonword_list_5,toithel,2.65,2.75,13.52,3.45
2,nonword_list_1,ligose,2.55,2.10,13.82,3.59
55,nonword_list_5,werdane,2.65,2.55,14.04,3.89
4,nonword_list_1,bermose,2.21,2.55,14.07,3.74
59,nonword_list_5,treinege,2.38,2.80,14.13,4.08
0,nonword_list_1,arfase,2.89,2.00,14.22,4.09
50,nonword_list_5,walsund,2.70,2.80,14.29,3.43
10,nonword_list_1,bulnete,2.44,2.65,14.34,3.68
25,nonword_list_3,heafnure,3.04,3.25,14.36,3.73


## List level: mean over the list's nonwords

In [2]:
nw.stimuli[["stimulus_id", "n_words", *SCORES]].round(2)

,stimulus_id,n_words,orth_trigram_surp,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20
0,nonword_list_1,12,2.57,2.98,14.81,4.78
1,nonword_list_2,12,2.86,3.18,15.00,4.90
2,nonword_list_3,12,2.74,3.24,15.08,4.98
3,nonword_list_4,12,2.66,3.72,15.41,5.65
4,nonword_list_5,12,2.74,3.15,14.90,4.76


## Compared with real words

In [3]:
both = pd.concat([nw.words.assign(kind="nonword"), ref.words.assign(kind="real word")])
both["n_letters"] = both["word"].str.lower().str.count("[a-z]")
both.groupby("kind")[[*SCORES, "n_letters"]].agg(["mean", "std"]).round(2)

orth_trigram_surp       orth_old20       orth_nbhd_surprisal_b20  \
                       mean   std       mean   std                    mean   
kind                                                                         
nonword                2.71  0.28       3.25  0.79                   15.04   
real word              2.52  0.69       1.24  0.40                   11.78   

                orth_nbhd_entropy_b20       n_letters        
            std                  mean   std      mean   std  
kind                                                         
nonword    0.66                  5.01  0.82      7.97  1.23  
real word  1.44                  2.26  0.83      3.93  1.58

In [4]:
# how the scores relate to each other and to length (Spearman, nonwords)
nw_words = both[both["kind"] == "nonword"]
nw_words[[*SCORES, "n_letters"]].corr(method="spearman").round(2)

,orth_trigram_surp,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20,n_letters
orth_trigram_surp,1.00,0.21,0.25,-0.02,-0.01
orth_old20,0.21,1.00,0.76,0.54,0.91
orth_nbhd_surprisal_b20,0.25,0.76,1.00,0.72,0.65
orth_nbhd_entropy_b20,-0.02,0.54,0.72,1.00,0.58
n_letters,-0.01,0.91,0.65,0.58,1.00


## Examples: most and least ill-formed nonwords

In [5]:
def show(df, label, col):
    print(label)
    for r in df.itertuples():
        print(f"  {getattr(r, col):6.2f}  {r.word}")

for col in ["orth_nbhd_surprisal_b20", "orth_trigram_surp"]:
    ranked = nw.words.sort_values(col, ascending=False)
    print(col)
    show(ranked.head(2), "  High (least English-like)", col)
    show(ranked.tail(2)[::-1], "  Low (most English-like)", col)
    print()

orth_nbhd_surprisal_b20
  High (least English-like)
   16.61  bisfuitful
   16.28  juilehoke
  Low (most English-like)
   13.44  honude
   13.52  toithel

orth_trigram_surp
  High (least English-like)
    3.58  ifenep
    3.25  lewnept
  Low (most English-like)
    2.10  breistel
    2.21  bermose



Short function words (*the*, *a*, *of*) are scored over only a few letters, so their means are noisy.

The phonotactic counterpart (BLICK, `blick_word`) needs phones. None of these nonwords is in CMUdict, so it is
empty until hand phone labels are given (see `data/examples/nonwords/output/missing_phones.csv`).